# DeBERTa-v3 PER fine-tuning (Label Studio -> token classification)

In [ ]:
import os

# Anchor the working directory so every relative path below resolves the same
# way regardless of where Jupyter was launched from.
os.chdir(r"D:\development\2026_python\2026_lb")
print(os.getcwd())

In [ ]:
import json
import numpy as np
import torch

from datasets import Dataset, DatasetDict
from evaluate import load as load_metric

import transformers
from transformers import (
    AutoModelForTokenClassification,
    AutoTokenizer,
    DataCollatorForTokenClassification,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
    set_seed,
)

print(f"torch        {torch.__version__}")
print(f"transformers {transformers.__version__}")
print(f"Intel GPU (XPU) available: {torch.xpu.is_available()}")

In [ ]:
# --- Configuration ---
TARGET_MODEL = "microsoft/deberta-v3-base"
RUN_ID = "2026sep9"

save_path = f"./{RUN_ID}_deberta-3_base"          # final fine-tuned model
checkpoint_dir = f"{save_path}/checkpoints"        # Trainer's per-epoch checkpoints

TRAIN_FILES = [
    "trainingdata/finetune/2026_march19_1403.json",
    "trainingdata/finetune/2026_march19_confidence.json",   # uncertain cases
]

TARGET_LABELS = {"PER"}
MAX_LENGTH = 512
STRIDE = 128          # window overlap; must exceed the longest entity in tokens
SEED = 42

num_train_epochs = 20
learning_rate = 1e-5
per_device_train_batch_size = 4
gradient_accumulation_steps = 4                    # effective batch = 16
early_stopping_patience = 4

# Seeds Python, NumPy and torch. TrainingArguments(seed=...) reseeds at train
# time as well; both are set so the run is reproducible end to end.
set_seed(SEED)

In [ ]:
# Explicit label order. Deriving this with sorted() would give
# ['B-PER', 'I-PER', 'O'] and put B-PER at index 0; NER convention (and the
# pretrained head's bias initialisation) expects the Outside class at 0.
all_tags = ["O", "B-PER", "I-PER"]

label_to_id = {tag: i for i, tag in enumerate(all_tags)}
id_to_label = {i: tag for tag, i in label_to_id.items()}

print(label_to_id)

In [ ]:
# One tokenizer instance for the whole notebook: it is the only thing that
# tokenizes anything, and it is what gets saved next to the trained model.
tokenizer = AutoTokenizer.from_pretrained(TARGET_MODEL, use_fast=True)

# The alignment below reads token offsets, which only the fast (Rust) tokenizers
# provide. Fail loudly here rather than silently mislabelling every example.
assert tokenizer.is_fast, "A fast tokenizer is required for return_offsets_mapping."

In [ ]:
def loaddata(paths=TRAIN_FILES):
    """Read and concatenate the Label Studio exports."""
    combined = []
    for path in paths:
        with open(path, "r", encoding="utf-8") as f:
            batch = json.load(f)
        print(f"Loaded {len(batch):>5} tasks from {path}")
        combined.extend(batch)

    print(f"Total tasks: {len(combined)}")
    return combined

In [ ]:
TRAILING_PUNCTUATION = ",.;:"


def extract_annotations(label_studio_data, target_labels=TARGET_LABELS):
    """Label Studio export -> [{"text": str, "entities": [{start, end, label}]}].

    No tokenizer involved. Annotations stay as character offsets into the
    original text, and are only turned into token labels later, in a single
    tokenization pass. This is what removes the old double-tokenization.
    """
    documents = []
    skipped_no_annotation = 0

    for task in label_studio_data:
        text = task["data"]["ParagraphText"]

        try:
            results = task["annotations"][0]["result"]
        except (IndexError, KeyError):
            skipped_no_annotation += 1
            continue

        entities = []
        for ann in results:
            value = ann["value"]
            label = value["labels"][0]
            if label not in target_labels:
                continue

            start, end = value["start"], value["end"]

            # Annotators frequently drag past the end of the name, catching the
            # trailing comma or full stop and sometimes a space after it. Trim
            # punctuation and whitespace together in one loop: doing punctuation
            # first and whitespace second would leave the comma in "Blond, ",
            # because the loop would stop on the space and never reach it.
            while end > start and (text[end - 1].isspace()
                                   or text[end - 1] in TRAILING_PUNCTUATION):
                end -= 1

            # Same at the front, so the span begins on a real character.
            while start < end and text[start].isspace():
                start += 1

            if start < end:
                entities.append({"start": start, "end": end, "label": label})

        # Label Studio exports can carry the same span twice (e.g. re-annotated
        # or overlapping drags). Deduplicate so one span produces one B- tag,
        # which keeps the coverage check below meaningful.
        entities = [dict(t) for t in {tuple(sorted(e.items())) for e in entities}]
        entities.sort(key=lambda e: (e["start"], e["end"]))

        documents.append({"text": text, "entities": entities})

    print(f"Documents kept: {len(documents)}")
    print(f"Skipped (no completed annotation): {skipped_no_annotation}")
    print(f"Total {'/'.join(sorted(target_labels))} spans: "
          f"{sum(len(d['entities']) for d in documents)}")
    return documents

In [ ]:
label_studio_data = loaddata()
documents = extract_annotations(label_studio_data)

# Spot-check one document: the char spans should slice cleanly out of the text.
example = next(d for d in documents if d["entities"])
print("\n" + example["text"][:200])
for ent in example["entities"][:5]:
    print(f"  [{ent['start']:4d}:{ent['end']:4d}] {ent['label']:<4} "
          f"{example['text'][ent['start']:ent['end']]!r}")

In [ ]:
raw_dataset = Dataset.from_list(documents)

split = raw_dataset.train_test_split(test_size=0.1, seed=SEED)
dataset_dict = DatasetDict({"train": split["train"], "validation": split["test"]})

print(dataset_dict)

In [ ]:
def tokenize_and_align_labels(examples):
    """Tokenize into overlapping windows, then project char spans onto tokens.

    Documents longer than MAX_LENGTH become several overlapping windows instead
    of being cut off, so entities past the limit are no longer silently dropped
    and left labelled O. Offsets from a fast tokenizer stay relative to the
    original text in every window, so the alignment below is unchanged.
    """
    tokenized = tokenizer(
        examples["text"],
        truncation=True,
        max_length=MAX_LENGTH,
        stride=STRIDE,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
    )
    # Maps each window back to the document it came from.
    sample_map = tokenized.pop("overflow_to_sample_mapping")

    batch_labels = []
    for i in range(len(tokenized["input_ids"])):
        text = examples["text"][sample_map[i]]
        entities = examples["entities"][sample_map[i]]
        offsets = tokenized["offset_mapping"][i]
        seq_ids = tokenized.sequence_ids(i)   # None on [CLS] / [SEP] / padding

        real = [j for j, sid in enumerate(seq_ids) if sid is not None]
        window_start = min(offsets[j][0] for j in real)
        window_end = max(offsets[j][1] for j in real)

        # Real tokens start as O; special tokens are -100 so the loss ignores them.
        labels = [-100 if sid is None else label_to_id["O"] for sid in seq_ids]

        for ent in entities:
            start, end, label = ent["start"], ent["end"], ent["label"]

            covered = []
            for j in real:
                tok_start, tok_end = offsets[j]

                # DeBERTa's SentencePiece folds the preceding space into the
                # token, so a token's offset can begin one character to the left
                # of the word it represents. Skip that leading whitespace before
                # testing overlap, and drop tokens that are whitespace only --
                # otherwise a bare space token sitting just before an entity
                # would take the B- tag that belongs to the first real token.
                while tok_start < tok_end and text[tok_start].isspace():
                    tok_start += 1
                if tok_start >= tok_end:
                    continue

                if tok_start < end and tok_end > start:
                    covered.append(j)

            if not covered:
                continue          # entity lies outside this window entirely

            if start < window_start or end > window_end:
                # Entity is cut by the window edge. Mask it rather than tagging
                # the visible fragment, which would teach the model that half a
                # name is a whole one. The STRIDE overlap guarantees the entity
                # appears complete in the neighbouring window.
                for j in covered:
                    labels[j] = -100
                continue

            for k, j in enumerate(covered):
                labels[j] = label_to_id[f"B-{label}" if k == 0 else f"I-{label}"]

        batch_labels.append(labels)

    tokenized["labels"] = batch_labels
    tokenized.pop("offset_mapping")   # not a model input
    return tokenized


# NOTE: the train/validation split above happens on whole documents, before
# windowing, so every window from a document stays on one side of the split.
# Windowing first would leak overlapping text across the split.
tokenized_datasets = dataset_dict.map(
    tokenize_and_align_labels,
    batched=True,
    remove_columns=raw_dataset.column_names,
)

print(tokenized_datasets)
print(f"Documents: {len(raw_dataset)}  ->  windows: "
      f"{len(tokenized_datasets['train']) + len(tokenized_datasets['validation'])}")

In [ ]:
# --- Coverage check -------------------------------------------------------
# Every annotated span should end up fully inside at least one window and so
# receive a real B- tag. An entity that never does is invisible to training,
# and worse, its tokens sit labelled O -- actively teaching the model that the
# name is not a name. This is the number to watch whenever the data, the
# tokenizer, MAX_LENGTH or STRIDE changes.
expected_spans = sum(len(d["entities"]) for d in documents)
covered_spans = 0
uncovered = []

for doc in documents:
    text, entities = doc["text"], doc["entities"]
    if not entities:
        continue

    enc = tokenizer(text, truncation=True, max_length=MAX_LENGTH, stride=STRIDE,
                    return_overflowing_tokens=True, return_offsets_mapping=True)

    # Character range each window spans, using the same rule as the labeller.
    window_bounds = []
    for i in range(len(enc["input_ids"])):
        seq_ids = enc.sequence_ids(i)
        offsets = enc["offset_mapping"][i]
        real = [j for j, sid in enumerate(seq_ids) if sid is not None]
        window_bounds.append((min(offsets[j][0] for j in real),
                              max(offsets[j][1] for j in real)))

    for ent in entities:
        if any(ws <= ent["start"] and ent["end"] <= we for ws, we in window_bounds):
            covered_spans += 1
        else:
            uncovered.append((len(enc["input_ids"]), ent,
                              text[ent["start"]:ent["end"]]))

print(f"Annotated spans in source : {expected_spans}")
print(f"Spans covered by a window : {covered_spans}")
print(f"Coverage: {covered_spans / expected_spans:.2%}")

if uncovered:
    print(f"\nNOTE: {len(uncovered)} span(s) never fall wholly inside a window.")
    print("With windowing enabled this means the span is longer than STRIDE, or")
    print("two annotated spans share a token. Examples:")
    for n_windows, ent, span in uncovered[:5]:
        print(f"  {span!r} at [{ent['start']}:{ent['end']}] "
              f"(document has {n_windows} window(s))")
else:
    print("\nAll annotated spans are fully represented in at least one window.")

In [ ]:
# --- Alignment check ------------------------------------------------------
# Print exactly what the model will see: its own tokens, with their labels.
for i in range(min(3, len(tokenized_datasets["train"]))):
    row = tokenized_datasets["train"][i]
    tokens = tokenizer.convert_ids_to_tokens(row["input_ids"])

    print(f"\n--- Sample {i} ---")
    for token, label_id in zip(tokens, row["labels"]):
        if label_id in (-100, label_to_id["O"]):
            continue
        print(f"  {token:<20} {id_to_label[label_id]}")

In [ ]:
metric = load_metric("seqeval")


def compute_metrics(eval_pred):
    """Trainer hands over (logits, label_ids); drop -100 positions before scoring."""
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    true_labels = [
        [id_to_label[l] for l in label if l != -100]
        for label in labels
    ]
    true_predictions = [
        [id_to_label[p] for p, l in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    results = metric.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

In [ ]:
model = AutoModelForTokenClassification.from_pretrained(
    TARGET_MODEL,
    num_labels=len(id_to_label),
    id2label=id_to_label,
    label2id=label_to_id,
)

data_collator = DataCollatorForTokenClassification(tokenizer)

In [ ]:
training_args = TrainingArguments(
    output_dir=checkpoint_dir,

    num_train_epochs=num_train_epochs,
    learning_rate=learning_rate,
    per_device_train_batch_size=per_device_train_batch_size,
    per_device_eval_batch_size=per_device_train_batch_size,
    gradient_accumulation_steps=gradient_accumulation_steps,
    weight_decay=0.01,
    max_grad_norm=1.0,                 # gradient clipping, was manual before
    lr_scheduler_type="linear",
    warmup_ratio=0.06,                 # scales with run length automatically

    # Non-fused AdamW. Recent transformers default optim to "adamw_torch_fused",
    # but torch._fused_adamw_ requires fp64 on the device, and Intel consumer
    # GPUs (Arc, integrated Xe) have no native fp64 -- the kernel is refused with
    # "Required aspect fp64 is not supported on the device". The non-fused path
    # is the same optimizer mathematically, just without the fused kernel.
    optim="adamw_torch",

    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    load_best_model_at_end=True,       # required by EarlyStoppingCallback
    metric_for_best_model="f1",
    greater_is_better=True,
    save_total_limit=2,

    # Memory saving; drop to False for a speed-up if the GPU has headroom.
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},

    # float32. DeBERTa's disentangled attention computes log-bucketed relative
    # positions, which has historically overflowed in reduced precision on XPU.
    # To re-verify: set bf16=True and watch the first two epochs of training
    # loss -- if it stays finite, keep it for the speed and memory win.
    bf16=False,
    fp16=False,

    seed=SEED,
    data_seed=SEED,
    report_to="none",
)

# Trainer places the model on XPU automatically when torch.xpu is available;
# no manual .to(device) and no ipex.optimize() wrapper.
print(f"Training device: {training_args.device}")

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=early_stopping_patience)],
)

train_result = trainer.train()

In [ ]:
# load_best_model_at_end=True means the in-memory model is already the best
# checkpoint by validation F1, so this saves that one, not merely the last.
trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)

print(f"Best checkpoint: {trainer.state.best_model_checkpoint}")
print(f"Best validation F1: {trainer.state.best_metric:.6f}")
print(f"Model saved to {save_path}")

In [ ]:
# Write the per-epoch report from Trainer's own log history.
metrics_file_path = os.path.join(save_path, "training_metrics_report.txt")

train_loss_by_epoch = {
    round(rec["epoch"]): rec["loss"]
    for rec in trainer.state.log_history if "loss" in rec
}

with open(metrics_file_path, "w", encoding="utf-8") as f:
    f.write(f"Model: {TARGET_MODEL}\n")
    f.write(f"Run:   {RUN_ID}\n")
    f.write(f"Seed:  {SEED}\n\n")

    for rec in trainer.state.log_history:
        if "eval_f1" not in rec:
            continue

        epoch = round(rec["epoch"])
        train_loss = train_loss_by_epoch.get(epoch)
        train_loss_text = f"{train_loss:.4f}" if train_loss is not None else "n/a"

        f.write(
            f"Epoch {epoch}:\n"
            f"  Training Loss:        {train_loss_text}\n"
            f"  Validation Loss:      {rec['eval_loss']:.4f}\n"
            f"  Validation Precision: {rec['eval_precision']:.6f}\n"
            f"  Validation Recall:    {rec['eval_recall']:.6f}\n"
            f"  Validation F1-Score:  {rec['eval_f1']:.6f}\n"
            f"  Validation Accuracy:  {rec['eval_accuracy']:.6f}\n\n"
        )

    f.write(f"Best validation F1: {trainer.state.best_metric:.6f}\n")

print(f"Training report saved to {metrics_file_path}")
print(open(metrics_file_path, encoding="utf-8").read())

In [ ]:
# End-to-end check: reload the saved model from disk exactly as downstream code
# would, and confirm it tags a held-out style sentence.
from transformers import pipeline

ner = pipeline(
    "token-classification",
    model=save_path,
    tokenizer=save_path,
    aggregation_strategy="simple",
)

for sentence in [
    "Nicholas de Suffok, William le Blond, Robert Heyrun.",
    "And that Robert Heyrun owed twenty shillings to the said William.",
]:
    print(f"\n{sentence}")
    for ent in ner(sentence):
        print(f"  {ent['entity_group']:<5} {ent['score']:.3f}  {ent['word']!r}")